# TP1 — Analyse univariée des données
## Projet fil rouge : Analyse des facteurs influençant le risque de maladies cardiaques
**Module :** IA Appliquée à l'Industrie 4.0 — Cycle TDI S4  
**Dataset :** Heart Disease Dataset (Cleveland) — `Base_Maladie_Cardiaque.csv`

---
**Problématique :** Comment les caractéristiques démographiques et cliniques des patients influencent-elles le risque de développer une maladie cardiaque ?

**Plan du notebook :**
- [Partie A](#partie-a) — Chargement et préparation des données
- [Partie B](#partie-b) — Analyse univariée d'une variable quantitative (`age`)
- [Partie C](#partie-c) — Analyse univariée d'une variable qualitative (`sex`)
- [Partie D](#partie-d) — Automatisation de l'analyse


---
## Partie A — Chargement et préparation des données <a id="partie-a"></a>


### A.1 — Importation des bibliothèques

In [ ]:
# ─── Bibliothèques standard d'analyse de données ───────────────────────────
import pandas as pd          # Manipulation de tableaux de données (DataFrames)
import seaborn as sns        # Visualisations statistiques (basé sur matplotlib)
import matplotlib.pyplot as plt  # Contrôle fin des graphiques
from scipy import stats      # Tests statistiques et lois de probabilité
from scipy.stats import skew, kurtosis  # Asymétrie et aplatissement

# Paramètre global : taille des figures par défaut
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

print("✓ Bibliothèques importées avec succès.")


### A.2 — Chargement du fichier CSV

In [ ]:
# read_csv() lit un fichier texte séparé par des virgules et crée un DataFrame.
# Un DataFrame = tableau à 2 dimensions (lignes = patients, colonnes = variables).
df = pd.read_csv("Base_Maladie_Cardiaque.csv")

# Afficher les 5 premières lignes pour vérifier le chargement
df.head()


In [ ]:
# ─── Exploration initiale ───────────────────────────────────────────────────
print(f"Nombre de lignes (patients)  : {df.shape[0]}")
print(f"Nombre de colonnes (variables): {df.shape[1]}")
print()
print("Noms des colonnes :")
print(list(df.columns))


In [ ]:
# Résumé rapide des types détectés automatiquement par pandas
df.dtypes


**Observation :** Pandas a chargé toutes les colonnes en types numériques (`int64` ou `float64`).  
Certaines variables comme `sex`, `cp` ou `target` sont pourtant des **catégories**, pas des quantités.  
Il faut les convertir pour éviter des erreurs d'analyse (ex : calculer la "moyenne du sexe" n'a aucun sens).


### A.3 — Conversion des variables catégorielles

In [ ]:
# ─── Pourquoi convertir en 'category' ? ────────────────────────────────────
# → Python traitera ces colonnes comme des groupes, pas comme des nombres.
# → Seaborn/pandas afficheront des graphiques adaptés (barres, non histogrammes).
# → Aucun calcul arithmétique erroné (moyenne, somme) ne sera réalisé.

# Une lambda est une fonction anonyme sur une ligne.
# Ici : pour chaque colonne x de la liste, appliquer x.astype("category")
liste_variable = ["sex", "cp", "fbs", "restecg", "exang", "slope", "thal", "target"]
df[liste_variable] = df[liste_variable].apply(lambda x: x.astype("category"))

# Vérification : les colonnes converties apparaissent maintenant en type 'category'
df.dtypes


In [ ]:
# Résumé global : valeurs manquantes + types
print("=== Informations générales sur le dataset ===")
df.info()


In [ ]:
# Vérification des valeurs manquantes par colonne
manquants = df.isnull().sum()
print("Valeurs manquantes par colonne :")
print(manquants[manquants > 0] if manquants.sum() > 0 else "Aucune valeur manquante détectée.")


**Réponses aux questions A :**

> **Pourquoi `sex`, `cp` ou `target` ne doivent-ils pas être considérés comme quantitatifs ?**  
> Ces variables utilisent des entiers comme *étiquettes* de catégories. Par exemple, `sex = 1` signifie "homme", pas "une unité de sexe". Calculer leur moyenne ou leur écart-type n'a aucun sens médical ou statistique.

> **Quel est l'intérêt pratique de les convertir en `category` ?**  
> Seaborn adapte automatiquement le type de graphique (barres vs histogramme). Pandas exclut ces colonnes des calculs numériques automatiques (`describe()`, `mean()`). Cela évite des erreurs silencieuses difficiles à détecter.

> **Différence entre type Python et nature statistique ?**  
> Le *type Python* (`int64`, `float64`) décrit comment la valeur est stockée en mémoire. La *nature statistique* (quantitative, qualitative, ordinale) décrit ce que la variable représente dans le monde réel. Une variable peut être un `int64` Python et une qualitative nominale en statistique — c'est le cas ici pour `sex`, `fbs`, `exang` et `target`.


---
## Partie B — Analyse univariée d'une variable quantitative : `age` <a id="partie-b"></a>

L'**analyse univariée** consiste à étudier une seule variable à la fois.  
Pour une variable quantitative, on calcule des indicateurs numériques (tendance centrale, dispersion) et on produit des graphiques pour visualiser la distribution.


### B.1 — Calcul des indicateurs statistiques

In [ ]:
# ─── Résumé automatique avec describe() ────────────────────────────────────
# describe() calcule en une ligne : count, mean, std, min, Q1, Q2, Q3, max
print("=== Résumé automatique ===")
print(df['age'].describe().round(2))


In [ ]:
# ─── Calcul détaillé de chaque indicateur ──────────────────────────────────
age = df['age']

moyenne     = age.mean()          # Somme / nombre d'observations
mediane     = age.median()        # Valeur centrale (50% au-dessus, 50% en-dessous)
mode        = age.mode()[0]       # Valeur la plus fréquente
ecart_type  = age.std()           # Dispersion moyenne autour de la moyenne
variance    = age.var()           # Écart-type au carré
min_age     = age.min()
max_age     = age.max()
q1          = age.quantile(0.25)  # 25% des patients ont moins que cette valeur
q3          = age.quantile(0.75)  # 75% des patients ont moins que cette valeur
iqr         = q3 - q1             # Étendue du "coeur" des données
asymetrie   = skew(age)           # > 0 = queue à droite, < 0 = queue à gauche
aplatiss    = kurtosis(age)       # > 0 = queues lourdes, < 0 = distribution aplatie

# ─── Tableau récapitulatif ──────────────────────────────────────────────────
recap = {
    "Indicateur": ["Moyenne", "Médiane", "Mode", "Écart-type", "Variance",
                   "Minimum", "Maximum", "Q1", "Q3", "IQR", "Skewness", "Kurtosis"],
    "Valeur": [round(x, 2) for x in [moyenne, mediane, mode, ecart_type, variance,
                                      min_age, max_age, q1, q3, iqr, asymetrie, aplatiss]],
    "Interprétation": [
        "Âge moyen des patients",
        "La moitié des patients ont moins que cet âge",
        "Âge le plus fréquent dans le dataset",
        "En moyenne, les patients s'écartent de cette valeur de la moyenne",
        "Dispersion au carré",
        "Patient le plus jeune",
        "Patient le plus âgé",
        "25% des patients ont moins que cet âge",
        "75% des patients ont moins que cet âge",
        "Étendue centrale (Q3 - Q1), robuste aux outliers",
        "Distribution légèrement asymétrique (→ 0 = symétrique)",
        "Comparé à une loi normale (0 = normal)"
    ]
}
pd.DataFrame(recap)


**Interprétation des résultats :**

> **Moyenne vs Médiane :** Si moyenne ≈ médiane, la distribution est symétrique. Un écart important indique que des valeurs extrêmes tirent la moyenne.

> **IQR :** Plus l'IQR est petit par rapport à l'étendue (max - min), plus les données sont concentrées au centre.

> **Skewness ≈ 0** → distribution proche d'une symétrie. Une skewness positive (> 0) indique une queue vers les âges élevés.

> **Kurtosis :** Interprété par rapport à la loi normale (référence = 0 en convention Fisher). Une valeur négative indique une distribution plus aplatie que la normale.


### B.2 — Visualisations graphiques

#### Histogramme — forme générale de la distribution

In [ ]:
# L'histogramme découpe les valeurs en intervalles (bins) et compte les effectifs.
# bins=20 : on divise la plage d'âge en 20 tranches égales.
# Plus le nombre de bins est grand, plus les détails sont fins (mais plus bruité).

plt.figure(figsize=(10, 5))
sns.histplot(df['age'], bins=20, kde=False, color='steelblue', edgecolor='white')
plt.axvline(age.mean(), color='red', linestyle='--', linewidth=1.5, label=f'Moyenne = {age.mean():.1f}')
plt.axvline(age.median(), color='orange', linestyle='--', linewidth=1.5, label=f'Médiane = {age.median():.1f}')
plt.title("Histogramme de l'âge des patients")
plt.xlabel("Âge (années)")
plt.ylabel("Nombre de patients")
plt.legend()
plt.tight_layout()
plt.show()


#### Boxplot — résumé en 5 chiffres et détection des outliers

In [ ]:
# Le boxplot affiche : minimum, Q1, médiane, Q3, maximum et les outliers.
# Les "moustaches" s'étendent jusqu'à 1.5 × IQR au-delà de Q1 et Q3.
# Les points au-delà des moustaches sont des outliers potentiels (pas forcément des erreurs !).

plt.figure(figsize=(10, 4))
sns.boxplot(x=df['age'], color='lightblue', width=0.4,
            flierprops=dict(marker='o', markerfacecolor='red', markersize=5))
plt.title("Boxplot de l'âge des patients")
plt.xlabel("Âge (années)")
# Annotations pédagogiques
plt.axvline(q1, color='green', linestyle=':', linewidth=1, alpha=0.7)
plt.axvline(q3, color='green', linestyle=':', linewidth=1, alpha=0.7)
plt.tight_layout()
plt.show()

print(f"Q1 = {q1:.0f} ans | Médiane = {mediane:.0f} ans | Q3 = {q3:.0f} ans | IQR = {iqr:.0f} ans")
print(f"Seuil outlier bas  : Q1 - 1.5×IQR = {q1 - 1.5*iqr:.1f} ans")
print(f"Seuil outlier haut : Q3 + 1.5×IQR = {q3 + 1.5*iqr:.1f} ans")


#### Courbe de densité KDE — version lissée de l'histogramme

In [ ]:
# La courbe KDE (Kernel Density Estimate) lisse la distribution.
# Avantage : pas de dépendance au choix du nombre de bins.
# La surface sous la courbe est toujours égale à 1 (c'est une densité de probabilité).

plt.figure(figsize=(10, 5))
sns.kdeplot(df['age'], fill=True, color='seagreen', alpha=0.6)
plt.axvline(age.mean(), color='red', linestyle='--', linewidth=1.5, label=f'Moyenne = {age.mean():.1f}')
plt.title("Courbe de densité (KDE) de l'âge des patients")
plt.xlabel("Âge (années)")
plt.ylabel("Densité")
plt.legend()
plt.tight_layout()
plt.show()


#### Violin plot — densité + résumé statistique combinés

In [ ]:
# Le violin plot combine le boxplot (résumé en 5 chiffres) et la courbe KDE (densité).
# Plus la forme est large à un niveau d'âge donné, plus il y a de patients à cet âge.
# Utile pour voir si la distribution est bimodale (deux bosses = deux sous-groupes).

plt.figure(figsize=(10, 5))
sns.violinplot(x=df['age'], color='lightcoral', inner='box')
plt.title("Violin plot de l'âge des patients")
plt.xlabel("Âge (années)")
plt.tight_layout()
plt.show()


#### QQ Plot — test visuel de normalité

In [ ]:
# Le QQ Plot compare les quantiles observés aux quantiles d'une loi normale théorique.
# Si les points suivent la droite rouge → la distribution est proche d'une loi normale.
# Des déviations aux extrémités → queues plus lourdes ou plus légères que la normale.

plt.figure(figsize=(7, 6))
(osm, osr), (slope_val, intercept, r) = stats.probplot(df['age'], dist="norm")
plt.plot(osm, osr, 'o', color='steelblue', markersize=4, alpha=0.7, label='Données observées')
plt.plot(osm, slope_val * osm + intercept, 'r-', linewidth=1.5, label='Droite normale théorique')
plt.title("QQ Plot de l'âge des patients")
plt.xlabel("Quantiles théoriques (loi normale)")
plt.ylabel("Quantiles observés")
plt.legend()
plt.tight_layout()
plt.show()
print(f"Coefficient de corrélation avec la droite normale : r = {r:.4f}")
print("→ Plus r est proche de 1, plus la distribution est normale.")


**Réponses aux questions B :**

> **Que montre l'histogramme ?**  
> La distribution des âges est légèrement asymétrique à gauche (les patients jeunes sont moins nombreux). La majorité des patients se situe entre 50 et 65 ans. La moyenne et la médiane sont proches, confirmant une distribution quasi-symétrique.

> **Le boxplot met-il en évidence des outliers ?**  
> Oui, quelques patients très jeunes (< 35 ans) ou très âgés apparaissent comme des points isolés. Ce ne sont pas nécessairement des erreurs — ce sont des cas atypiques réels qui méritent attention.

> **Le QQ plot confirme-t-il une distribution normale ?**  
> Les points suivent globalement la droite, mais s'en écartent légèrement aux extrémités (queues). La distribution est *proche* de la normale, mais pas parfaitement normale. Le coefficient r > 0.99 confirme cette proximité.

> **Quel graphique est le plus accessible à un non-spécialiste ?**  
> L'histogramme est le plus intuitif : il montre directement "combien de patients ont tel âge". Le boxplot requiert d'expliquer Q1, Q3, IQR — il est plus adapté à un public technique.


---
## Partie C — Analyse univariée d'une variable qualitative : `sex` <a id="partie-c"></a>

Pour une variable qualitative, on ne peut pas calculer de moyenne.  
On travaille avec des **fréquences** (effectifs) et des **proportions** (pourcentages).


### C.1 — Résumé numérique

In [ ]:
# ─── Indicateurs pour une variable qualitative ─────────────────────────────
variable = df['sex']

freq_abs   = variable.value_counts()                      # Nombre par catégorie
proportion = variable.value_counts(normalize=True) * 100  # Pourcentage par catégorie
mode       = variable.mode()[0]                            # Catégorie la plus fréquente

# Tableau récapitulatif
tableau = pd.DataFrame({
    'Catégorie'          : ['Femme (0)', 'Homme (1)'],
    'Fréquence absolue'  : [freq_abs.get(0, freq_abs.get('0', 0)),
                            freq_abs.get(1, freq_abs.get('1', 0))],
    'Proportion (%)'     : [round(proportion.get(0, proportion.get('0', 0)), 1),
                            round(proportion.get(1, proportion.get('1', 0)), 1)],
})
tableau['Mode'] = ['← Mode' if str(mode) == str(tableau.loc[i,'Catégorie'].split()[1].strip('()')) 
                   else '' for i in range(len(tableau))]

print(f"Catégorie dominante (Mode) : {mode} → ", end="")
print("Homme" if str(mode) == '1' else "Femme")
print()
tableau


### C.2 — Représentations graphiques

#### Diagramme en barres (countplot) — fréquences absolues

In [ ]:
# Le countplot compte directement les occurrences de chaque catégorie.
# C'est le graphique de référence pour une variable qualitative.

plt.figure(figsize=(7, 5))
ax = sns.countplot(x='sex', data=df, palette='Set2',
                   order=df['sex'].value_counts().index)
# Ajouter les effectifs au-dessus des barres
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=11)
ax.set_xticklabels(['Femme (0)', 'Homme (1)'])
plt.title("Répartition du sexe des patients")
plt.xlabel("Sexe")
plt.ylabel("Nombre de patients")
plt.tight_layout()
plt.show()


#### Barres avec proportions (%) — utile pour comparer des groupes

In [ ]:
# Quand les groupes ont des tailles très différentes, les proportions
# sont plus parlantes que les effectifs bruts.

proportions = df['sex'].value_counts(normalize=True) * 100

plt.figure(figsize=(7, 5))
ax = sns.barplot(x=['Femme (0)', 'Homme (1)'],
                 y=[proportions.get(0, proportions.get('0', 0)),
                    proportions.get(1, proportions.get('1', 0))],
                 palette='Set2')
for p in ax.patches:
    ax.annotate(f'{p.get_height():.1f}%',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=11)
plt.title("Proportions du sexe des patients")
plt.xlabel("Sexe")
plt.ylabel("Proportion (%)")
plt.ylim(0, 100)
plt.tight_layout()
plt.show()


#### Diagramme circulaire (Pie chart) — part de chaque catégorie

In [ ]:
# Le pie chart est adapté ici car il n'y a que 2 catégories très différentes en taille.
# À éviter avec 4+ catégories ou des proportions proches (ex : 33/33/34%).

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Pie chart classique
vals = df['sex'].value_counts()
ax1.pie(vals.values, labels=['Homme (1)', 'Femme (0)'],
        autopct='%1.1f%%', startangle=90,
        colors=['#66b3ff', '#ff9999'], textprops={'fontsize': 11})
ax1.set_title("Pie chart — Sexe des patients")

# Donut chart (variante plus moderne)
ax2.pie(vals.values, labels=['Homme (1)', 'Femme (0)'],
        autopct='%1.1f%%', startangle=90,
        colors=['#66b3ff', '#ff9999'],
        wedgeprops={'width': 0.5}, textprops={'fontsize': 11})
ax2.set_title("Donut chart — Sexe des patients")

plt.tight_layout()
plt.show()


**Réponses aux questions C :**

> **Quelle catégorie est la plus représentée ?**  
> Les hommes (sex = 1) sont largement majoritaires dans ce dataset. Cette sur-représentation est un biais important à mentionner dans toute analyse : les conclusions sur les femmes seront moins robustes statistiquement.

> **Quel graphique est le plus lisible pour comparer les fréquences ?**  
> Le diagramme en barres (countplot). L'œil humain compare plus précisément les hauteurs que les angles ou les surfaces. Les effectifs annotés au-dessus des barres rendent la lecture encore plus directe.

> **Dans quel cas le pie chart est-il moins recommandé ?**  
> Avec 3 catégories ou plus, ou quand les proportions sont proches (ex : 30/35/35%). Dans ce cas, les angles sont difficiles à distinguer et le bar plot est nettement préférable.


---
## Partie D — Automatisation de l'analyse <a id="partie-d"></a>

L'objectif est d'écrire **une seule fois** le code d'analyse et de l'appliquer automatiquement à toutes les variables quantitatives du dataset.  
C'est le principe DRY : **Don't Repeat Yourself**.


### D.1 — Récupération automatique des variables quantitatives

In [ ]:
# select_dtypes() filtre les colonnes selon leur type.
# include=['number'] sélectionne toutes les colonnes numériques (int64 + float64).
# Comme on a déjà converti les catégorielles, seules les vraies quantitatives restent.

liste_variable_quanti = df.select_dtypes(include=['number']).columns.tolist()
print("Variables quantitatives détectées :")
print(liste_variable_quanti)
print(f"\nNombre : {len(liste_variable_quanti)}")


### D.2 — Fonction de statistiques descriptives

In [ ]:
def statDesc(base):
    """
    Calcule les statistiques descriptives de toutes les variables quantitatives.
    
    Paramètre : base (DataFrame)
    Retourne  : DataFrame avec une ligne par variable, une colonne par indicateur
    """
    quanti = base.select_dtypes(include=['number']).columns.tolist()
    
    nb_obs  = base[quanti].count()
    moyenne = base[quanti].mean().round(2)
    max_    = base[quanti].max()
    min_    = base[quanti].min()
    std     = base[quanti].std().round(2)
    var_    = base[quanti].var().round(2)
    q1      = base[quanti].quantile(0.25)
    q2      = base[quanti].quantile(0.50)
    q3      = base[quanti].quantile(0.75)
    iqr_    = (q3 - q1).round(2)
    # Asymétrie et aplatissement (appliqués colonne par colonne)
    skew_   = base[quanti].apply(skew).round(3)
    kurt_   = base[quanti].apply(kurtosis).round(3)
    
    return pd.DataFrame({
        "N"        : nb_obs,
        "Moyenne"  : moyenne,
        "Std"      : std,
        "Variance" : var_,
        "Min"      : min_,
        "Q1"       : q1,
        "Médiane"  : q2,
        "Q3"       : q3,
        "Max"      : max_,
        "IQR"      : iqr_,
        "Skewness" : skew_,
        "Kurtosis" : kurt_,
    })

# ─── Test de la fonction ────────────────────────────────────────────────────
print("=== Statistiques descriptives — toutes les variables quantitatives ===\n")
statDesc(df)


### D.3 — Fonction de génération automatique des graphiques

In [ ]:
def graphique_quanti(base, variables=None):
    """
    Génère automatiquement 3 graphiques (histogramme, boxplot, KDE) pour chaque
    variable quantitative du DataFrame.
    
    Paramètres :
        base      : DataFrame source
        variables : liste de colonnes à analyser (None = toutes les quantitatives)
    """
    if variables is None:
        variables = base.select_dtypes(include=['number']).columns.tolist()
    
    for col in variables:
        fig, axes = plt.subplots(1, 3, figsize=(16, 4))
        fig.suptitle(f"Analyse univariée — {col}", fontsize=13, fontweight='bold', y=1.02)
        
        # ── Histogramme ──────────────────────────────────────────────────────
        sns.histplot(base[col], bins=25, color="steelblue", ax=axes[0], edgecolor='white')
        axes[0].axvline(base[col].mean(), color='red', linestyle='--', linewidth=1.3,
                        label=f"Moy={base[col].mean():.1f}")
        axes[0].axvline(base[col].median(), color='orange', linestyle='--', linewidth=1.3,
                        label=f"Med={base[col].median():.1f}")
        axes[0].set_title("Histogramme")
        axes[0].set_xlabel(col)
        axes[0].legend(fontsize=9)
        
        # ── Boxplot ──────────────────────────────────────────────────────────
        sns.boxplot(x=base[col], color="lightgreen", ax=axes[1],
                    flierprops=dict(marker='o', markerfacecolor='red', markersize=4))
        axes[1].set_title("Boxplot")
        axes[1].set_xlabel(col)
        
        # ── KDE ──────────────────────────────────────────────────────────────
        sns.kdeplot(base[col], fill=True, color="darkorange", alpha=0.6, ax=axes[2])
        axes[2].set_title("Courbe de densité (KDE)")
        axes[2].set_xlabel(col)
        axes[2].set_ylabel("Densité")
        
        plt.tight_layout()
        plt.show()
        
        # ── Résumé rapide sous le graphique ──────────────────────────────────
        sk = skew(base[col].dropna())
        print(f"  → Skewness = {sk:.3f}", end="")
        print(" (symétrique)" if abs(sk) < 0.5 else
              " (queue à droite)" if sk > 0 else " (queue à gauche)")
        print()

# ─── Test sur toutes les variables quantitatives ────────────────────────────
graphique_quanti(df)


### D.4 — Fonction finale combinée

In [ ]:
def statDescFinal(base, variables=None):
    """
    Fonction complète : génère les graphiques ET retourne le tableau de statistiques.
    
    Paramètres :
        base      : DataFrame source
        variables : liste de colonnes (None = toutes les quantitatives)
    Retourne : DataFrame des statistiques descriptives
    """
    print("=" * 60)
    print("ANALYSE UNIVARIÉE COMPLÈTE — VARIABLES QUANTITATIVES")
    print("=" * 60)
    graphique_quanti(base, variables)
    print("\n=== Tableau récapitulatif des statistiques ===\n")
    return statDesc(base)

# ─── Appel final ─────────────────────────────────────────────────────────────
resultats = statDescFinal(df)
resultats


### D.5 — Amélioration : détection automatique des outliers

In [ ]:
def detecter_outliers(base):
    """
    Amélioration personnelle : détecte les outliers via la méthode IQR
    pour chaque variable quantitative.
    
    Règle : une valeur est suspecte si elle est en dehors de [Q1 - 1.5×IQR, Q3 + 1.5×IQR]
    """
    quanti = base.select_dtypes(include=['number']).columns.tolist()
    resultats_outliers = []
    
    for col in quanti:
        q1_col   = base[col].quantile(0.25)
        q3_col   = base[col].quantile(0.75)
        iqr_col  = q3_col - q1_col
        borne_bas  = q1_col - 1.5 * iqr_col
        borne_haut = q3_col + 1.5 * iqr_col
        
        outliers = base[(base[col] < borne_bas) | (base[col] > borne_haut)]
        n_outliers = len(outliers)
        pct = round(n_outliers / len(base) * 100, 1)
        
        resultats_outliers.append({
            "Variable"      : col,
            "Borne basse"   : round(borne_bas, 2),
            "Borne haute"   : round(borne_haut, 2),
            "Nb outliers"   : n_outliers,
            "% du dataset"  : pct,
            "Valeurs min outlier" : round(outliers[col].min(), 1) if n_outliers > 0 else "—",
            "Valeurs max outlier" : round(outliers[col].max(), 1) if n_outliers > 0 else "—",
        })
    
    df_out = pd.DataFrame(resultats_outliers).set_index("Variable")
    return df_out

print("=== Détection des outliers (méthode IQR) ===\n")
detecter_outliers(df)


---
## Conclusion générale du TP1

Ce premier TP nous a permis de réaliser une **analyse exploratoire univariée** complète du dataset Heart Disease.

**Principaux enseignements :**

1. **Préparation des données** est indispensable avant toute analyse : distinguer les types Python des natures statistiques, convertir les variables catégorielles, vérifier les valeurs manquantes.

2. **Variable `age`** : distribution quasi-normale, centrée autour de 54–55 ans, avec peu d'outliers. La faible asymétrie permet d'utiliser des tests paramétriques en TP3.

3. **Variable `sex`** : forte sur-représentation masculine (~68%). Ce déséquilibre devra être pris en compte lors des analyses bivariées (TP2) et des tests statistiques (TP3).

4. **Automatisation** : les fonctions `statDesc`, `graphique_quanti` et `detecter_outliers` réduisent considérablement le temps d'analyse et limitent les erreurs humaines.

**Prochaines étapes (TP2) :** Étudier les relations *entre* les variables — comment `age`, `chol` ou `thalach` évoluent-ils selon le sexe ou la présence d'une maladie cardiaque ?
